In [1]:
from __future__ import annotations

import os
from getpass import getpass
from typing import Any

import weaviate
import weaviate.classes as wvc

from weaviate.classes.init import Auth
from weaviate.classes.query import Filter, MetadataQuery

In [2]:
if "WEAVIATE_URL" not in os.environ:
    os.environ["WEAVIATE_URL"] = getpass("WEAVIATE_URL: ")

if "WEAVIATE_API_KEY" not in os.environ:
    os.environ["WEAVIATE_API_KEY"] = getpass("WEAVIATE_API_KEY: ")

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [3]:
client = weaviate.connect_to_weaviate_cloud(
    cluster_url=os.environ["WEAVIATE_URL"],
    auth_credentials=Auth.api_key(os.environ["WEAVIATE_API_KEY"]),
    headers={
        "X-OpenAI-Api-Key": os.environ["OPENAI_API_KEY"],
    },
)

print("Client ready:", client.is_ready())

Client ready: True


In [4]:
COLLECTION_NAME = "SafeRagDocument"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

docs = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="title",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="content",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="topic",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="source",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="trusted",
            data_type=wvc.config.DataType.BOOL,
        ),
    ],
)

print("Created collection:", COLLECTION_NAME)

Created collection: SafeRagDocument


In [5]:
safe_rag_data = [
    {
        "title": "Weaviate Cloud Authentication",
        "content": (
            "To connect to Weaviate Cloud, use the cluster URL, authenticate with "
            "the Weaviate API key and pass provider headers such as the OpenAI API key "
            "when using text2vec_openai or generative_openai."
        ),
        "topic": "cloud",
        "source": "cloud_notebook",
        "trusted": True,
    },
    {
        "title": "Collections and Objects",
        "content": (
            "A Weaviate collection is similar to a SQL table. An object is similar "
            "to a row. Properties are similar to columns, and UUIDs identify objects."
        ),
        "topic": "data_model",
        "source": "crud_notebook",
        "trusted": True,
    },
    {
        "title": "Vector Search",
        "content": (
            "Vector search uses embeddings to find semantically similar objects. "
            "The query is vectorized and compared with stored object vectors."
        ),
        "topic": "vector_search",
        "source": "vector_search_notebook",
        "trusted": True,
    },
    {
        "title": "BM25 Keyword Search",
        "content": (
            "BM25 is keyword-based search. It works well for exact terms, product names, "
            "technical words, categories and identifiers."
        ),
        "topic": "bm25",
        "source": "hybrid_search_notebook",
        "trusted": True,
    },
    {
        "title": "Hybrid Search",
        "content": (
            "Hybrid search combines BM25 keyword search with vector search. "
            "The alpha parameter controls the balance between keyword-based ranking "
            "and semantic vector ranking."
        ),
        "topic": "hybrid_search",
        "source": "hybrid_search_notebook",
        "trusted": True,
    },
    {
        "title": "Generative Search",
        "content": (
            "Generative search retrieves relevant objects from Weaviate and sends them "
            "to a language model using single_prompt or grouped_task to generate answers."
        ),
        "topic": "generative_search",
        "source": "generative_search_notebook",
        "trusted": True,
    },
    {
        "title": "Focused RAG",
        "content": (
            "Focused RAG works best when the collection contains only relevant chunks, "
            "for example Weaviate notebooks and Markdown concept notes, instead of an entire "
            "unrelated repository."
        ),
        "topic": "rag",
        "source": "focused_rag_notebook",
        "trusted": True,
    },
    {
        "title": "Named Vectors",
        "content": (
            "Named vectors allow one object to have multiple vector spaces, such as "
            "title_vector and description_vector, so search can target different semantic fields."
        ),
        "topic": "named_vectors",
        "source": "named_vectors_notebook",
        "trusted": True,
    },
    {
        "title": "Multi-Tenancy",
        "content": (
            "Multi-tenancy allows one shared collection schema to isolate data for different "
            "customers, companies, users or organizations."
        ),
        "topic": "multi_tenancy",
        "source": "multi_tenancy_notebook",
        "trusted": True,
    },
    {
        "title": "JSONL Snapshot",
        "content": (
            "A JSONL snapshot stores one JSON object per line. It can be used for "
            "application-level export, inspection and restore workflows."
        ),
        "topic": "snapshot",
        "source": "snapshot_notebook",
        "trusted": True,
    },
]

In [6]:
result = docs.data.insert_many(safe_rag_data)

if result.errors:
    print("Import errors:")
    print(result.errors)
else:
    print("Import completed.")

Import completed.


In [7]:
response = docs.query.fetch_objects(
    limit=20,
    return_properties=[
        "title",
        "topic",
        "source",
        "trusted",
    ],
)

for obj in response.objects:
    print("UUID:", obj.uuid)
    print(obj.properties)
    print("-" * 80)

UUID: 008c687d-7f6b-43ff-8c8a-1271d0c0e4f2
{'source': 'hybrid_search_notebook', 'topic': 'bm25', 'title': 'BM25 Keyword Search', 'trusted': True}
--------------------------------------------------------------------------------
UUID: 20a02f21-54fa-45a4-8e95-984dc8dec44c
{'source': 'named_vectors_notebook', 'topic': 'named_vectors', 'title': 'Named Vectors', 'trusted': True}
--------------------------------------------------------------------------------
UUID: 3df5cc4a-819b-4aaa-a8ad-e281cc5b9222
{'source': 'cloud_notebook', 'topic': 'cloud', 'title': 'Weaviate Cloud Authentication', 'trusted': True}
--------------------------------------------------------------------------------
UUID: 4e47ced1-2120-42b6-b641-f753afdca6b7
{'source': 'vector_search_notebook', 'topic': 'vector_search', 'title': 'Vector Search', 'trusted': True}
--------------------------------------------------------------------------------
UUID: 5703d56f-9a59-4126-b1f5-a80dfae02e49
{'source': 'crud_notebook', 'topic': 'da

In [8]:
response = docs.query.near_text(
    query="how does hybrid search work in Weaviate",
    limit=5,
    return_properties=[
        "title",
        "content",
        "topic",
        "source",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.39704978466033936
Title: Hybrid Search
Topic: hybrid_search
Content: Hybrid search combines BM25 keyword search with vector search. The alpha parameter controls the balance between keyword-based ranking and semantic vector ranking.
--------------------------------------------------------------------------------
Distance: 0.489269495010376
Title: Generative Search
Topic: generative_search
Content: Generative search retrieves relevant objects from Weaviate and sends them to a language model using single_prompt or grouped_task to generate answers.
--------------------------------------------------------------------------------
Distance: 0.5613446235656738
Title: Weaviate Cloud Authentication
Topic: cloud
Content: To connect to Weaviate Cloud, use the cluster URL, authenticate with the Weaviate API key and pass provider headers such as the OpenAI API key when using text2vec_openai or generative_openai.
----------------------------------------------------------------------------

In [9]:
response = docs.query.near_text(
    query="how to optimize Kubernetes ingress controller for production",
    limit=5,
    return_properties=[
        "title",
        "content",
        "topic",
        "source",
    ],
    return_metadata=MetadataQuery(distance=True),
)

for obj in response.objects:
    print("Distance:", obj.metadata.distance)
    print("Title:", obj.properties["title"])
    print("Topic:", obj.properties["topic"])
    print("Content:", obj.properties["content"])
    print("-" * 80)

Distance: 0.7650874853134155
Title: Weaviate Cloud Authentication
Topic: cloud
Content: To connect to Weaviate Cloud, use the cluster URL, authenticate with the Weaviate API key and pass provider headers such as the OpenAI API key when using text2vec_openai or generative_openai.
--------------------------------------------------------------------------------
Distance: 0.8378763794898987
Title: Focused RAG
Topic: rag
Content: Focused RAG works best when the collection contains only relevant chunks, for example Weaviate notebooks and Markdown concept notes, instead of an entire unrelated repository.
--------------------------------------------------------------------------------
Distance: 0.8513357639312744
Title: Generative Search
Topic: generative_search
Content: Generative search retrieves relevant objects from Weaviate and sends them to a language model using single_prompt or grouped_task to generate answers.
---------------------------------------------------------------------------

In [10]:
def evaluate_retrieval_quality(
    objects,
    *,
    max_distance: float = 0.35,
    min_results: int = 1,
) -> dict[str, Any]:
    if len(objects) < min_results:
        return {
            "is_good_enough": False,
            "reason": "Not enough retrieved objects.",
            "best_distance": None,
            "result_count": len(objects),
        }

    distances = [
        obj.metadata.distance
        for obj in objects
        if obj.metadata.distance is not None
    ]

    if not distances:
        return {
            "is_good_enough": False,
            "reason": "No distance metadata available.",
            "best_distance": None,
            "result_count": len(objects),
        }

    best_distance = min(distances)

    if best_distance > max_distance:
        return {
            "is_good_enough": False,
            "reason": f"Best distance {best_distance:.4f} is above threshold {max_distance}.",
            "best_distance": best_distance,
            "result_count": len(objects),
        }

    return {
        "is_good_enough": True,
        "reason": "Retrieval quality is acceptable.",
        "best_distance": best_distance,
        "result_count": len(objects),
    }

In [11]:
response = docs.query.near_text(
    query="what is hybrid search in Weaviate",
    limit=5,
    return_properties=[
        "title",
        "content",
        "topic",
        "source",
    ],
    return_metadata=MetadataQuery(distance=True),
)

quality = evaluate_retrieval_quality(
    response.objects,
    max_distance=0.35,
    min_results=1,
)

quality

{'is_good_enough': False,
 'reason': 'Best distance 0.4045 is above threshold 0.35.',
 'best_distance': 0.404472291469574,
 'result_count': 5}

In [12]:
response = docs.query.near_text(
    query="how to optimize Kubernetes ingress controller for production",
    limit=5,
    return_properties=[
        "title",
        "content",
        "topic",
        "source",
    ],
    return_metadata=MetadataQuery(distance=True),
)

quality = evaluate_retrieval_quality(
    response.objects,
    max_distance=0.35,
    min_results=1,
)

quality

{'is_good_enough': False,
 'reason': 'Best distance 0.7651 is above threshold 0.35.',
 'best_distance': 0.7650694847106934,
 'result_count': 5}

In [13]:
def build_context(objects) -> str:
    lines = []

    for number, obj in enumerate(objects, start=1):
        props = obj.properties

        lines.extend(
            [
                f"Source {number}",
                f"Title: {props['title']}",
                f"Topic: {props['topic']}",
                f"Source name: {props['source']}",
                f"Content: {props['content']}",
                "",
            ]
        )

    return "\n".join(lines)

In [14]:
CONTEXT_COLLECTION_NAME = "SafeAnswerContext"

if client.collections.exists(CONTEXT_COLLECTION_NAME):
    client.collections.delete(CONTEXT_COLLECTION_NAME)

answer_contexts = client.collections.create(
    name=CONTEXT_COLLECTION_NAME,
    vector_config=wvc.config.Configure.Vectors.text2vec_openai(
        model="text-embedding-3-small",
    ),
    generative_config=wvc.config.Configure.Generative.openai(
        model="gpt-4o-mini",
    ),
    properties=[
        wvc.config.Property(
            name="question",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="context",
            data_type=wvc.config.DataType.TEXT,
        ),
        wvc.config.Property(
            name="quality_reason",
            data_type=wvc.config.DataType.TEXT,
        ),
    ],
)

print("Created collection:", CONTEXT_COLLECTION_NAME)

Created collection: SafeAnswerContext


In [15]:
def safe_answer(
    question: str,
    *,
    max_distance: float = 0.35,
    retrieval_limit: int = 5,
) -> None:
    retrieval_response = docs.query.near_text(
        query=question,
        filters=Filter.by_property("trusted").equal(True),
        limit=retrieval_limit,
        return_properties=[
            "title",
            "content",
            "topic",
            "source",
            "trusted",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    quality = evaluate_retrieval_quality(
        retrieval_response.objects,
        max_distance=max_distance,
        min_results=1,
    )

    print("QUESTION:")
    print(question)

    print("\nRETRIEVAL QUALITY:")
    print(quality)

    print("\nRETRIEVED SOURCES:")
    for obj in retrieval_response.objects:
        print(
            "-",
            obj.properties["title"],
            "| topic:",
            obj.properties["topic"],
            "| distance:",
            obj.metadata.distance,
        )

    if not quality["is_good_enough"]:
        print("\nSAFE ANSWER:")
        print(
            "I do not have enough reliable information in the Weaviate knowledge base "
            "to answer this question."
        )
        return

    context = build_context(retrieval_response.objects)

    answer_contexts.data.insert(
        {
            "question": question,
            "context": context,
            "quality_reason": quality["reason"],
        }
    )

    answer_response = answer_contexts.generate.near_text(
        query=question,
        single_prompt=(
            "Answer the question using only the provided context.\n\n"
            "Question: {question}\n\n"
            "Context:\n{context}\n\n"
            "Retrieval quality note: {quality_reason}\n\n"
            "If the context does not contain the answer, say that the knowledge base "
            "does not contain enough information. Do not invent facts."
        ),
        limit=1,
        return_properties=[
            "question",
            "context",
            "quality_reason",
        ],
    )

    print("\nSAFE ANSWER:")
    for obj in answer_response.objects:
        print(obj.generated)

In [16]:
safe_answer(
    "What is hybrid search in Weaviate and what does alpha control?",
    max_distance=0.35,
)

QUESTION:
What is hybrid search in Weaviate and what does alpha control?

RETRIEVAL QUALITY:
{'is_good_enough': False, 'reason': 'Best distance 0.4229 is above threshold 0.35.', 'best_distance': 0.42290568351745605, 'result_count': 5}

RETRIEVED SOURCES:
- Hybrid Search | topic: hybrid_search | distance: 0.42290568351745605
- Generative Search | topic: generative_search | distance: 0.5781018733978271
- Weaviate Cloud Authentication | topic: cloud | distance: 0.5975502729415894
- BM25 Keyword Search | topic: bm25 | distance: 0.6504919528961182
- Vector Search | topic: vector_search | distance: 0.6741987466812134

SAFE ANSWER:
I do not have enough reliable information in the Weaviate knowledge base to answer this question.


In [17]:
safe_answer(
    "How should I build focused RAG over notebooks and Markdown notes?",
    max_distance=0.35,
)

QUESTION:
How should I build focused RAG over notebooks and Markdown notes?

RETRIEVAL QUALITY:
{'is_good_enough': True, 'reason': 'Retrieval quality is acceptable.', 'best_distance': 0.23722362518310547, 'result_count': 5}

RETRIEVED SOURCES:
- Focused RAG | topic: rag | distance: 0.23722362518310547
- Generative Search | topic: generative_search | distance: 0.5454205274581909
- Collections and Objects | topic: data_model | distance: 0.5579724311828613
- JSONL Snapshot | topic: snapshot | distance: 0.5932105779647827
- Multi-Tenancy | topic: multi_tenancy | distance: 0.6331008672714233

SAFE ANSWER:
To build focused RAG (Retrieval-Augmented Generation) over notebooks and Markdown notes, ensure that your collection includes only relevant chunks, such as Weaviate notebooks and Markdown concept notes, rather than an entire unrelated repository.


In [18]:
safe_answer(
    "How do I deploy a full production AI platform with monitoring, billing and Kubernetes?",
    max_distance=0.35,
)

QUESTION:
How do I deploy a full production AI platform with monitoring, billing and Kubernetes?

RETRIEVAL QUALITY:
{'is_good_enough': False, 'reason': 'Best distance 0.6708 is above threshold 0.35.', 'best_distance': 0.6708294749259949, 'result_count': 5}

RETRIEVED SOURCES:
- Weaviate Cloud Authentication | topic: cloud | distance: 0.6708294749259949
- Multi-Tenancy | topic: multi_tenancy | distance: 0.7812510132789612
- Generative Search | topic: generative_search | distance: 0.8237417340278625
- Hybrid Search | topic: hybrid_search | distance: 0.8553574085235596
- Collections and Objects | topic: data_model | distance: 0.8741706609725952

SAFE ANSWER:
I do not have enough reliable information in the Weaviate knowledge base to answer this question.


In [19]:
question = "How do I deploy a full production AI platform with monitoring, billing and Kubernetes?"

for threshold in [0.25, 0.35, 0.45, 0.55]:
    print("\nTHRESHOLD:", threshold)
    print("=" * 80)

    safe_answer(
        question,
        max_distance=threshold,
    )


THRESHOLD: 0.25
QUESTION:
How do I deploy a full production AI platform with monitoring, billing and Kubernetes?

RETRIEVAL QUALITY:
{'is_good_enough': False, 'reason': 'Best distance 0.6708 is above threshold 0.25.', 'best_distance': 0.6708294749259949, 'result_count': 5}

RETRIEVED SOURCES:
- Weaviate Cloud Authentication | topic: cloud | distance: 0.6708294749259949
- Multi-Tenancy | topic: multi_tenancy | distance: 0.7812510132789612
- Generative Search | topic: generative_search | distance: 0.8237417340278625
- Hybrid Search | topic: hybrid_search | distance: 0.8553574085235596
- Collections and Objects | topic: data_model | distance: 0.8741706609725952

SAFE ANSWER:
I do not have enough reliable information in the Weaviate knowledge base to answer this question.

THRESHOLD: 0.35
QUESTION:
How do I deploy a full production AI platform with monitoring, billing and Kubernetes?

RETRIEVAL QUALITY:
{'is_good_enough': False, 'reason': 'Best distance 0.6708 is above threshold 0.35.', '

In [20]:
def safe_answer_payload(
    question: str,
    *,
    max_distance: float = 0.35,
    retrieval_limit: int = 5,
) -> dict[str, Any]:
    retrieval_response = docs.query.near_text(
        query=question,
        filters=Filter.by_property("trusted").equal(True),
        limit=retrieval_limit,
        return_properties=[
            "title",
            "content",
            "topic",
            "source",
            "trusted",
        ],
        return_metadata=MetadataQuery(distance=True),
    )

    quality = evaluate_retrieval_quality(
        retrieval_response.objects,
        max_distance=max_distance,
        min_results=1,
    )

    sources = [
        {
            "title": obj.properties["title"],
            "topic": obj.properties["topic"],
            "source": obj.properties["source"],
            "distance": obj.metadata.distance,
        }
        for obj in retrieval_response.objects
    ]

    if not quality["is_good_enough"]:
        return {
            "answered": False,
            "answer": (
                "I do not have enough reliable information in the Weaviate knowledge base "
                "to answer this question."
            ),
            "quality": quality,
            "sources": sources,
        }

    context = build_context(retrieval_response.objects)

    answer_contexts.data.insert(
        {
            "question": question,
            "context": context,
            "quality_reason": quality["reason"],
        }
    )

    answer_response = answer_contexts.generate.near_text(
        query=question,
        single_prompt=(
            "Answer the question using only the provided context.\n\n"
            "Question: {question}\n\n"
            "Context:\n{context}\n\n"
            "Do not invent facts. If the answer is missing, say that the knowledge base "
            "does not contain enough information."
        ),
        limit=1,
        return_properties=[
            "question",
            "context",
        ],
    )

    generated_answer = None

    for obj in answer_response.objects:
        generated_answer = obj.generated

    return {
        "answered": True,
        "answer": generated_answer,
        "quality": quality,
        "sources": sources,
    }

In [21]:
payload = safe_answer_payload(
    "What are named vectors in Weaviate?",
    max_distance=0.35,
)

payload

{'answered': False,
 'answer': 'I do not have enough reliable information in the Weaviate knowledge base to answer this question.',
 'quality': {'is_good_enough': False,
  'reason': 'Best distance 0.3775 is above threshold 0.35.',
  'best_distance': 0.37748706340789795,
  'result_count': 5},
 'sources': [{'title': 'Named Vectors',
   'topic': 'named_vectors',
   'source': 'named_vectors_notebook',
   'distance': 0.37748706340789795},
  {'title': 'Weaviate Cloud Authentication',
   'topic': 'cloud',
   'source': 'cloud_notebook',
   'distance': 0.5277702212333679},
  {'title': 'Vector Search',
   'topic': 'vector_search',
   'source': 'vector_search_notebook',
   'distance': 0.5747467875480652},
  {'title': 'Generative Search',
   'topic': 'generative_search',
   'source': 'generative_search_notebook',
   'distance': 0.5979318618774414},
  {'title': 'Collections and Objects',
   'topic': 'data_model',
   'source': 'crud_notebook',
   'distance': 0.6424866914749146}]}

In [22]:
payload = safe_answer_payload(
    "What is the best tax strategy for a limited company?",
    max_distance=0.35,
)

payload

{'answered': False,
 'answer': 'I do not have enough reliable information in the Weaviate knowledge base to answer this question.',
 'quality': {'is_good_enough': False,
  'reason': 'Best distance 0.8506 is above threshold 0.35.',
  'best_distance': 0.8505645990371704,
  'result_count': 5},
 'sources': [{'title': 'Multi-Tenancy',
   'topic': 'multi_tenancy',
   'source': 'multi_tenancy_notebook',
   'distance': 0.8505645990371704},
  {'title': 'Focused RAG',
   'topic': 'rag',
   'source': 'focused_rag_notebook',
   'distance': 0.8995025157928467},
  {'title': 'BM25 Keyword Search',
   'topic': 'bm25',
   'source': 'hybrid_search_notebook',
   'distance': 0.9016991853713989},
  {'title': 'Hybrid Search',
   'topic': 'hybrid_search',
   'source': 'hybrid_search_notebook',
   'distance': 0.9149903059005737},
  {'title': 'JSONL Snapshot',
   'topic': 'snapshot',
   'source': 'snapshot_notebook',
   'distance': 0.9258577227592468}]}

In [23]:
def print_answer_with_sources(payload: dict[str, Any]) -> None:
    print("ANSWERED:", payload["answered"])
    print("\nANSWER:")
    print(payload["answer"])

    print("\nQUALITY:")
    print(payload["quality"])

    print("\nSOURCES:")
    for source in payload["sources"]:
        print(
            "-",
            source["title"],
            "| topic:",
            source["topic"],
            "| source:",
            source["source"],
            "| distance:",
            source["distance"],
        )

In [24]:
payload = safe_answer_payload(
    "Explain the difference between vector search, BM25 and hybrid search.",
    max_distance=0.35,
)

print_answer_with_sources(payload)

ANSWERED: True

ANSWER:
Vector search uses embeddings to find semantically similar objects by vectorizing the query and comparing it with stored object vectors. BM25, on the other hand, is a keyword-based search technique that excels at finding exact terms, such as product names and identifiers. Hybrid search combines both BM25 and vector search, utilizing an alpha parameter to balance between keyword-based ranking and semantic vector ranking.

QUALITY:
{'is_good_enough': True, 'reason': 'Retrieval quality is acceptable.', 'best_distance': 0.288371205329895, 'result_count': 5}

SOURCES:
- Hybrid Search | topic: hybrid_search | source: hybrid_search_notebook | distance: 0.288371205329895
- BM25 Keyword Search | topic: bm25 | source: hybrid_search_notebook | distance: 0.431427001953125
- Vector Search | topic: vector_search | source: vector_search_notebook | distance: 0.5239226222038269
- Named Vectors | topic: named_vectors | source: named_vectors_notebook | distance: 0.6401336193084717

In [25]:
payload = safe_answer_payload(
    "Explain how to configure PostgreSQL replication.",
    max_distance=0.35,
)

print_answer_with_sources(payload)

ANSWERED: False

ANSWER:
I do not have enough reliable information in the Weaviate knowledge base to answer this question.

QUALITY:
{'is_good_enough': False, 'reason': 'Best distance 0.8118 is above threshold 0.35.', 'best_distance': 0.8118330240249634, 'result_count': 5}

SOURCES:
- Multi-Tenancy | topic: multi_tenancy | source: multi_tenancy_notebook | distance: 0.8118330240249634
- Weaviate Cloud Authentication | topic: cloud | source: cloud_notebook | distance: 0.8421194553375244
- Collections and Objects | topic: data_model | source: crud_notebook | distance: 0.8448615074157715
- Generative Search | topic: generative_search | source: generative_search_notebook | distance: 0.8639858961105347
- JSONL Snapshot | topic: snapshot | source: snapshot_notebook | distance: 0.9024738073348999


In [26]:
client.close()